In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys

from projects.droid_maniskill.droid_pick_cube_env import DROIDPickCubeEnv
from robot_wm.inference.robot.simulated_franka_maniskill import NewSimulatedRobot
from hydra.utils import instantiate
from einops import rearrange
from omegaconf import OmegaConf
import torch

**Load LAM**

In [ ]:

snapshot_dir = "data/experiments/latent_action_models/dino_st_lam_vq_action_decoder-1024/2025-07-08/00-25-50/0"
config_path = f"{snapshot_dir}/.hydra"
print(config_path)

cfg = OmegaConf.load(config_path + "/config.yaml")

In [ ]:
sys.path.append("projects/latent_action_models")
model = instantiate(cfg.model)

val_dataloader = instantiate(cfg.val_data_loader)
model = model.cuda().eval()

disgusting hack

In [ ]:
from math import prod
for batch in val_dataloader:
    obs = batch["rgb"].cuda()
    z = model._forward_tokenizer_encode(obs)
    z_shape = prod(z.shape[2:])
    break
print(f"z shape: {z_shape}")

In [ ]:
from robot_wm.modeling.modules.mlp import MLP
from torch import nn
import torch.nn.functional as F

class Policy(nn.Module):
    def __init__(self, rgb_tokenizer, input_dim, codebook, decoder, quantizer, num_actions):
        super().__init__()
        self.rgb_tokenizer = rgb_tokenizer
        self.decoder = decoder
        self.quantizer = quantizer
        codebook_size = codebook.shape[0]
        self.model = MLP(
            input_dim=input_dim,
            output_dim=512,
            hidden_dim=[512, 512],
        )
        self.heads = nn.ModuleList([nn.Linear(512, codebook_size) for _ in range(num_actions)])
        self.register_buffer("codebook", codebook)

    def _forward_tokenizer_encode(self, obs):
        if obs.dim() == 4:
            obs = obs.unsqueeze(0)
        x = self.rgb_tokenizer.encode(obs)
        x = rearrange(x, "N T d h w -> N T h w d")
        x = rearrange(x, "N T h w d -> N T (h w d)")
        return x

    def _generate_decoded_action(self, obs):
        logits = self.forward(obs)
        logits = torch.stack(logits, dim=-2)
        # print(f"logits shape: {logits.shape}")
        
        # logits to codebook indices without quantization
        indices = logits.argmax(dim=-1)  # (N, T, num_actions)
        codes = self.codebook[indices]
        # print(f"indices shape: {indices.shape}, codes shape: {codes.shape}")
        codes = rearrange(codes, "N T A D -> N T (A D)")

        return self.decoder(codes)


        #return self.decoder(zhat)

        # quantize the logits


    def forward(self, obs):
        # print(f"obs shape: {obs.shape} ")
        z = self._forward_tokenizer_encode(obs)
        # print(f"z shape: {z.shape}")
        h = self.model(z)
        logits = [head(h) for head in self.heads]
        return logits

num_actions = model.inverse_model.output_proj.query.num_embeddings
pi = Policy(
    rgb_tokenizer=model.rgb_tokenizer,
    input_dim=z_shape,
    codebook=model.quantizer.embed,
    decoder=model.action_decoder,
    quantizer=model.quantizer,
    num_actions=num_actions
).cuda()

In [ ]:

def get_inverse_action(obs):
    print(f"obs shape: {obs.shape}")
    z = model._forward_tokenizer_encode(obs)
    print(f"z shape: {z.shape}")
    with torch.no_grad():
        action_labels = model._forward_inverse_model(z)
    vq_output = model.quantizer(action_labels)
    return vq_output["indices"]

In [ ]:
optimizer = torch.optim.Adam(pi.parameters(), lr=1e-4)
opt_steps = 1
from tqdm import tqdm
for step in tqdm(range(opt_steps)):
    for batch in val_dataloader:
        optimizer.zero_grad()

        obs = batch["rgb"].cuda()
        actions = batch["actions"].cuda()
        # print(f"obs shape: {obs.shape}, actions shape: {actions.shape}")

        action_labels = get_inverse_action(obs)
        # print(f"action_labels shape: {action_labels.shape}")

        decoded_actions = pi._generate_decoded_action(obs)
        # print(f"decoded_actions shape: {decoded_actions.shape}")
        
        latent_actions = pi(obs)
        latent_action_shapes = [la.shape for la in latent_actions]
        # print(f"latent_action shapes: {latent_action_shapes}")
        
        loss = 0
        for k in range(len(latent_actions)):
            # print(f"latent action {k} shape: {latent_actions[k].shape}, action_labels shape: {action_labels[..., k].shape}")
            la = rearrange(latent_actions[k], "N T d -> N d T")
            loss += F.cross_entropy(la, action_labels[..., k])
        loss /= len(latent_actions)

        print(f"loss: {loss.item()}")

        loss.backward()
        optimizer.step()

Create env
---------

In [ ]:
robot_config_dict = {"style_seed": 0}
robot = NewSimulatedRobot(robot_config_dict)

from robot_wm.inference.robot.base import (
    Robot,
    RobotEEDeltaAction,
    RobotObs,
    RobotState,
) 
import numpy as np
init_state = RobotState(
        joints=np.array(
            [
                2.0078e-03,
                3.7137e-01,
                3.6554e-02,
                -2.0464e+00,
                9.5134e-02,
                2.4619e+00,
                7.4044e-01
            ]
        )
    )
obs = robot.reset_to_state(init_state)

obs = robot.get_last_observation()
obs.show()
print(robot.env.action_space.sample())

Sim evaluation
------

In [ ]:
# get only one image

def obs_to_img(obs):
    obs = obs["sensor_data"]['left_camera']['rgb']
    return rearrange(obs, "B H W C -> B C H W")/255.0

# for batch in val_dataloader:
#     obs = batch["rgb"].cuda()
#     print(f"data obs shape: {obs.shape}, obs dtype: {obs.dtype}")
#     print(f" data obs min: {obs.min()}, data obs max: {obs.max()}")
#     break

num_runs = 2
average_reward = []
success_count = 0
for run in range(num_runs):
    # print(f"Run {run + 1}/{num_runs}")
    obs, _ = robot.env.reset()
    # obs = robot.get_last_observation()
    done = False
    episode_reward = 0
    step_count = 0
    while not done:
        obs = obs_to_img(obs)
        # print(f"obs shape: {obs.shape}, obs dtype: {obs.dtype}")
        # print(f" obs min: {obs.min()}, data obs max: {obs.max()}")

        action = pi._generate_decoded_action(obs)[0, ..., :7]
        # print(f"action shape: {action.shape}")
        act = action.cpu().detach().numpy()
        act = robot.env.action_space.sample()
        obs, reward, terminated, truncated, info = robot.env.step(act)
        robo_obs = robot.get_last_observation()
        robo_obs.show()

        done = terminated or truncated
        success = info.get("success", False)
        if success:
            success_count += 1
            print(f"Success in run {run + 1}!")
        
        episode_reward += reward
        average_reward.append(reward)
        step_count += 1
        if step_count == robot.env._max_episode_steps or success:
            print(f"Run {run + 1} reached max steps: {robot.env._max_episode_steps}")
            done = True
    
    print(f"Total reward for run {run + 1}: {episode_reward.item()}")

success_rate = success_count / num_runs if num_runs > 0 else 0
average_reward = np.mean(average_reward)

print(f"Average episodic reward over {num_runs} runs: {average_reward}")
print(f"Success rate: {success_rate:.2f}")
